In [ ]:
# Feature Engineering

In [ ]:
"""
What is feature engineering?
Feature engineering means creating new, meaningful information from existing data.
For example, your orders table may contain the purchase date and delivery date. 
It does not necessarily have a column telling you how many days delivery took. We can calculate that new feature in Python.
"""

In [1]:
# Install the required Python libraries
!pip install pandas numpy matplotlib seaborn scipy sqlalchemy pymysql

In [ ]:
"""
Library	and Purpose
pandas - Read SQL data and manipulate tables
numpy -	Numerical calculations
matplotlib - Create charts
seaborn - Create statistical charts
scipy - Statistical analysis
sqlalchemy - Connect Python to MySQL
pymysql - MySQL database driver

"""

In [2]:
# Import the libraries
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sqlalchemy import create_engine
from urllib.parse import quote_plus

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="notebook")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [ ]:
"""
Explanation:
- import pandas as pd: allows you to use commands such as pd.read_sql() and pd.DataFrame().
- import numpy as np: gives you numerical tools and missing-value handling.
- matplotlib.pyplot as plt: creates graphs.
- seaborn as sns: creates histograms, boxplots, heatmaps, and other statistical graphs.
- from scipy import stats: provides statistical tools, including Q-Q plots.
- create_engine: establishes a connection to MySQL.
- quote_plus: safely encodes special characters in your database password when constructing the connection URL.
- warnings.filterwarnings("ignore"): hides warnings. During debugging, it is better to remove this line so warnings remain visible.
- sns.set_theme(...): makes the charts easier to read.
- pd.set_option(...): adjusts how tables and numbers appear in notebook output.
"""

In [3]:
# Connect Jupyter Notebook to your MySQL database
import pandas as pd
from sqlalchemy import create_engine, URL

connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="Arul@nitha08",
    host="localhost",
    port=3306,
    database="olist_db"
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    print("MySQL connected successfully!")

MySQL connected successfully!


In [10]:
# 7.1 Bring the base data from SQL

orders = pd.read_sql("""
    SELECT o.order_id, o.customer_id, c.customer_unique_id, c.customer_state, c.customer_city,
           o.order_status, o.order_purchase_timestamp, o.order_approved_at,
           o.order_delivered_carrier_date, o.order_delivered_customer_date,
           o.order_estimated_delivery_date
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
""", engine)

date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

order_totals = pd.read_sql("""
    SELECT order_id,
           COUNT(*)                  AS item_count,
           COUNT(DISTINCT seller_id) AS seller_count,
           SUM(price)                AS items_value,
           SUM(freight_value)        AS freight_value,
           SUM(total_value)          AS order_value
    FROM order_items
    GROUP BY order_id
""", engine)

payments = pd.read_sql("""
    SELECT order_id,
           SUM(payment_value)        AS payment_value,
           MAX(payment_installments) AS max_installments,
           COUNT(*)                  AS payment_count
    FROM order_payments
    GROUP BY order_id
""", engine)

main_payment = pd.read_sql("""
    SELECT order_id, payment_type AS main_payment_type
    FROM order_payments
    WHERE payment_sequential = 1
""", engine)

reviews = pd.read_sql("""
    SELECT order_id, AVG(review_score) AS review_score
    FROM order_reviews
    GROUP BY order_id
""", engine)

print(orders.shape, order_totals.shape, payments.shape, main_payment.shape, reviews.shape)

(99441, 11) (98666, 6) (99437, 4) (99357, 2) (98669, 2)


In [11]:
# 7.2 Order-level features (total order value, delivery days, delivery delay)
# Total order value is order_value, which was summed in SQL above.
order_df = (orders
            .merge(order_totals, on="order_id", how="left")
            .merge(payments, on="order_id", how="left")
            .merge(main_payment, on="order_id", how="left")
            .merge(reviews, on="order_id", how="left"))

# Delivery days
order_df["delivery_days"] = (
    (order_df["order_delivered_customer_date"] - order_df["order_purchase_timestamp"])
    .dt.total_seconds() / 86400
)

# Delivery delay (positive = late)
order_df["delivery_delay_days"] = (
    (order_df["order_delivered_customer_date"] - order_df["order_estimated_delivery_date"])
    .dt.total_seconds() / 86400
)
order_df["is_late"] = np.where(order_df["delivery_delay_days"].isna(), np.nan,
                               (order_df["delivery_delay_days"] > 0).astype(float))

# Time features for trend analysis
order_df["purchase_month"] = order_df["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
order_df["purchase_dow"]   = order_df["order_purchase_timestamp"].dt.day_name()
order_df["purchase_hour"]  = order_df["order_purchase_timestamp"].dt.hour

# Keep real sales only
orders_valid = order_df[~order_df["order_status"].isin(["canceled", "unavailable"])
                        & order_df["order_value"].notna()].copy()
delivered = orders_valid[orders_valid["delivery_days"].notna()].copy()

print("all orders:", len(order_df), "| valid sales orders:", len(orders_valid), "| delivered:", len(delivered))
orders_valid[["order_id", "order_value", "delivery_days", "delivery_delay_days", "is_late"]].head()

all orders: 99441 | valid sales orders: 98199 | delivered: 96470


,order_id,order_value,delivery_days,delivery_delay_days,is_late
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,7.614421,-8.011250,0.0
1,00018f77f2f0320c557190d7a144bdd3,259.83,16.216181,-2.330278,0.0
2,000229ec398224ef6ca0657da4fc703e,216.87,7.948437,-13.444954,0.0
3,00024acbcdf0a6daa1e931b038114c75,25.78,6.147269,-5.435660,0.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,25.114352,-15.303808,0.0


In [12]:
# 7.3 Customer features (order count, total spending, average order value, repeat indicator)
# Customers are grouped on customer_unique_id, because customer_id changes with every order.
customer_features = (orders_valid.groupby("customer_unique_id")
    .agg(customer_order_count=("order_id", "nunique"),
         customer_total_spend=("order_value", "sum"),
         first_order_date=("order_purchase_timestamp", "min"),
         last_order_date=("order_purchase_timestamp", "max"),
         customer_state=("customer_state", "first"))
    .reset_index())

customer_features["avg_order_value"]     = customer_features["customer_total_spend"] / customer_features["customer_order_count"]
customer_features["is_repeat_customer"]  = (customer_features["customer_order_count"] > 1).astype(int)

print("customers:", len(customer_features))
print("repeat customers: {:.2f}%".format(customer_features["is_repeat_customer"].mean() * 100))
customer_features.head()

customers: 94983
repeat customers: 3.04%


,customer_unique_id,customer_order_count,customer_total_spend,first_order_date,last_order_date,customer_state,avg_order_value,is_repeat_customer
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,2018-05-10 10:56:27,2018-05-10 10:56:27,SP,141.90,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,2018-05-07 11:11:27,2018-05-07 11:11:27,SP,27.19,0
2,0000f46a3911fa3c0805444483337064,1,86.22,2017-03-10 21:05:03,2017-03-10 21:05:03,SC,86.22,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,2017-10-12 20:29:41,2017-10-12 20:29:41,PA,43.62,0
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,2017-11-14 19:45:42,2017-11-14 19:45:42,SP,196.89,0


In [13]:
# 7.4 Seller features (revenue, order count)
sellers = pd.read_sql("SELECT seller_id, seller_city, seller_state FROM sellers", engine)
items = pd.read_sql("""
    SELECT order_id, order_item_id, product_id, seller_id, price, freight_value, total_value
    FROM order_items
""", engine)

items_valid = items[items["order_id"].isin(orders_valid["order_id"])].copy()
items_valid = items_valid.merge(
    orders_valid[["order_id", "review_score", "delivery_delay_days"]], on="order_id", how="left")

seller_features = (items_valid.groupby("seller_id")
    .agg(seller_revenue=("price", "sum"),
         seller_order_count=("order_id", "nunique"),
         seller_items_sold=("order_id", "size"),
         seller_avg_item_price=("price", "mean"),
         seller_avg_review=("review_score", "mean"),
         seller_avg_delay_days=("delivery_delay_days", "mean"))
    .reset_index()
    .merge(sellers, on="seller_id", how="left"))

seller_features["seller_avg_order_value"] = seller_features["seller_revenue"] / seller_features["seller_order_count"]

print("sellers:", len(seller_features))
seller_features.sort_values("seller_revenue", ascending=False).head(10)

sellers: 3053


,seller_id,seller_revenue,seller_order_count,seller_items_sold,seller_avg_item_price,seller_avg_review,seller_avg_delay_days,seller_city,seller_state,seller_avg_order_value
849,4869f7a5dfa277a7dca6462dcf3b52b2,229237.63,1131,1155,198.474139,4.125545,-10.453161,guariba,SP,202.685791
1000,53243585a1d6dc2643021fd1853d8905,222776.05,358,410,543.356220,4.075980,-10.877354,lauro de freitas,BA,622.279469
873,4a3ca9315b744ce9f8e9374361493884,200326.12,1804,1985,100.919960,3.804592,-8.954641,ibitinga,SP,111.045521
2984,fa1c13f2614d7b5c4749cbc52fecda94,192842.13,584,585,329.644667,4.344234,-9.612764,sumare,SP,330.209127
1515,7c67e1448b00f6e969d365cea6b010ab,187923.89,982,1364,137.774113,3.345133,-10.832570,itaquaquecetuba,SP,191.368523
1540,7e93a43ef30c4f03f38b393420bc753a,172583.88,332,336,513.642500,4.238806,-11.485531,barueri,SP,519.830964
2611,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1314,1551,103.311779,4.067314,-10.527563,piracicaba,SP,121.945639
1485,7a67c85e85bb2ce8582c35f2203ad736,141715.54,1159,1170,121.124393,4.236865,-11.160569,sao paulo,SP,122.273978
191,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,915,1428,97.316912,3.846045,-10.388772,sao paulo,SP,151.878197
1799,955fee9216a65b617aa5c0531780ce60,135159.70,1286,1498,90.226769,4.052490,-9.626675,sao paulo,SP,105.100855


In [14]:
# 7.5 Attach customer features back to the order table
orders_valid = orders_valid.merge(
    customer_features[["customer_unique_id", "customer_order_count",
                       "customer_total_spend", "is_repeat_customer"]],
    on="customer_unique_id", how="left")
delivered = orders_valid[orders_valid["delivery_days"].notna()].copy()

orders_valid[["order_value", "freight_value", "item_count", "delivery_days",
              "delivery_delay_days", "customer_order_count",
              "customer_total_spend"]].describe().T

,count,mean,std,min,25%,50%,75%,max
order_value,98199.0,160.241215,219.111166,9.590000,61.960000,105.280000,176.730000,13664.080000
freight_value,98199.0,22.822292,21.645764,0.000000,13.860000,17.190000,24.050000,1794.960000
item_count,98199.0,1.141570,0.537556,1.000000,1.000000,1.000000,1.000000,21.000000
delivery_days,96470.0,12.558217,9.546156,0.533414,6.766204,10.217477,15.720182,209.628611
delivery_delay_days,96470.0,-11.178126,10.184354,-146.016123,-16.244065,-11.948102,-6.389815,188.975081
customer_order_count,98199.0,1.076844,0.383260,1.000000,1.000000,1.000000,1.000000,16.000000
customer_total_spend,98199.0,171.286611,233.902231,9.590000,64.090000,111.210000,189.080000,13664.080000


In [15]:
# 7.6 Optional: save the feature tables back to MySQL
# This creates two new tables and leaves the original tables untouched.
customer_features.to_sql("customer_features", engine, if_exists="replace", index=False, chunksize=5000)
seller_features.to_sql("seller_features", engine, if_exists="replace", index=False, chunksize=5000)
print("saved: customer_features, seller_features")

saved: customer_features, seller_features
